# Step 0 — data cleaning + zero-shot MedGemma baseline (thin notebook: it only calls scripts)
**Settings:** Accelerator = GPU T4 x2, Internet = on. **Add-ons → Secrets:** `HF_TOKEN` (access to MedGemma + ReXGradient-160K).
**Inputs to attach:** `raddar/chest-xrays-indiana-university`, your private `rexgradient-160k-test-images`
(made once with section 0 below), and the R2Gen IU-Xray `annotation.json` when you have it.
MIMIC is disabled until an official copy (PhysioNet) is available.

In [ ]:
# Code: clone the repo (github.com/Vinith-44/Radioreport), or attach it as a Kaggle dataset and copy it here instead.
!git clone -q --depth 1 https://github.com/Vinith-44/Radioreport /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q -r requirements.txt

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["RUN_NOTE"] = "step0 kaggle"   # free-text note logged with every run

## 0. ONE TIME: fetch the official ReXGradient test images from Hugging Face (~10 GB kept, 155 GB streamed)
Run in a separate session (no GPU needed). When it finishes: **Save Version** → open the version's **Output** →
**New Dataset** named `rexgradient-160k-test-images`, keep it **Private** (license: no redistribution).
Then skip this section in all later runs.

In [ ]:
!python scripts/step0_fetch_rexgradient.py --smoke   # ~2 min: proves streaming + extraction work
!python scripts/step0_fetch_rexgradient.py           # full: prints GB streamed, speed and ETA
!rm -rf /kaggle/working/rexgradient-160k-test-images/smoke_deid_png

**Resuming in a new session?** `/kaggle/working` is wiped between sessions. Attach the previous version's output as an input
and restore it, so generation continues where it stopped:

In [ ]:
PREV = None  # e.g. "/kaggle/input/step0-run-v1/project"
if PREV:
    !mkdir -p /kaggle/working/project && cp -rn {PREV}/outputs {PREV}/data /kaggle/working/project/

## 1. Check paths and columns, run the unit tests

In [ ]:
!python scripts/step0_prepare_data.py --config configs/step0.yaml --inspect
!python -m pytest tests -q

## 2. Smoke run (~50 studies per dataset) — inspect the outputs before the full run

In [ ]:
!python scripts/step0_prepare_data.py --config configs/step0.yaml --smoke
!python scripts/step0_zeroshot_generate.py --model medgemma-1.5 --run-name zeroshot --smoke
!python scripts/step0_zeroshot_generate.py --model medgemma-1   --run-name zeroshot --smoke
!python scripts/step0_score.py --run-name zeroshot --smoke

## 3. Full run
fp32 weights are sharded over both T4s, so the two models run one after the other.
Each command can be re-run after a crash or timeout and continues where it stopped.

In [ ]:
!python scripts/step0_prepare_data.py --config configs/step0.yaml
!cat /kaggle/working/project/data/manifest.json

In [ ]:
!python scripts/step0_zeroshot_generate.py --model medgemma-1.5 --run-name zeroshot
!python scripts/step0_zeroshot_generate.py --model medgemma-1   --run-name zeroshot

In [ ]:
!python scripts/step0_score.py --run-name zeroshot
!cat /kaggle/working/project/outputs/RESULTS.md